In [1]:
import torch
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Training transformations
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4), # Note: Changed 32 to 28 to match FashionMNIST natively, unless you specifically want 32x32!
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # Changed to 1 channel
])

# Testing transformations (No augmentation here!)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # Changed to 1 channel
])

In [3]:
# Load FashionMNIST dataset
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 60000
Test dataset size: 10000


In [4]:
# Define the optimized CNN architecture
class OptimizedFashionCNN(nn.Module):
    def __init__(self):
        super(OptimizedFashionCNN, self).__init__()
        
        # Block 1: 1 Channel In -> 32 Channels Out
        # Padding=1 keeps the image size exactly the same after convolution
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Block 2: 32 Channels In -> 64 Channels Out (The Funnel Shape)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.25) # Lower dropout for conv layers
        
        # The exact mathematical size after two 2x2 poolings on a 28x28 image is 7x7.
        # 64 channels * 7 height * 7 width = 3136
        self.fc1 = nn.Linear(in_features=3136, out_features=128)
        self.dropout_fc = nn.Dropout(0.5) # Standard 50% dropout for dense layers
        self.fc2 = nn.Linear(in_features=128, out_features=10) # 10 Output classes
        
    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool(x)
        
        # Block 2
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool(x)
        x = self.dropout(x)
        
        # Flatten exactly
        x = x.view(-1, 64 * 7 * 7)
        
        # Fully Connected Block
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        
        return x
    
model = OptimizedFashionCNN()
print(model)

OptimizedFashionCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout_fc): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [5]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

Epoch [1/10], Loss: 0.9227
Epoch [2/10], Loss: 0.7221
Epoch [3/10], Loss: 0.6616
Epoch [4/10], Loss: 0.6158
Epoch [5/10], Loss: 0.5883
Epoch [6/10], Loss: 0.5659
Epoch [7/10], Loss: 0.5467
Epoch [8/10], Loss: 0.5300
Epoch [9/10], Loss: 0.5114
Epoch [10/10], Loss: 0.5011


In [8]:
# Evaluate the model on the test set
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')


Test Accuracy: 87.89%
